# Caso C · 04 Isolation Forest + Autoencoder

> _Tutorial · Caso de uso: **C — Anomalías HVAC** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Entrenar Isolation Forest (no supervisado) y un Autoencoder MLP simple. Comparar AUC cuando reservamos las etiquetas para validación.


## 2. Qué se aprende

- Isolation Forest: hiperparámetros y score.
- Autoencoder MLP: reconstruction error como score.
- Cuándo no supervisado es mejor que supervisado.
- Threshold tuning con percentiles.


## 3. Contexto del caso de uso

El equipo C entrega un detector que opera en producción sin etiquetas — porque en CENTINELA+ los fallos reales son raros.


## 4. Relación con CENTINELA+

El detector se conecta como tool al chatbot Caso H (`check_hvac_anomaly`).


## 5. Relación con Medallion

Oro: modelo entrenado.


## 6. Datos de entrada

Oro features Caso C.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Reusamos las features.


In [ ]:
parquet_path = ROOT / "output" / "case_C" / "hvac_features.parquet"
if parquet_path.exists():
    X = pd.read_parquet(parquet_path)
else:
    df, _ = mocks.make_lbnl_fdd_rtu_mock()
    X = pd.DataFrame({
        "dt_supply_return": df["RA_TEMP"] - df["SA_TEMP"],
        "valve": df["CCV"],
        "fan": df["FAN_STATE"],
        "is_fault": df["is_fault"],
    }, index=df["timestamp"]).dropna()
print(X.shape)


## 10. Exploración paso a paso

Split: entrenar Isolation Forest sobre lo que llamamos 'normal' (puede contener trazas; no rompe).


In [ ]:
y = X.pop("is_fault").astype(int)
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.1, random_state=SEED, n_estimators=200)
iso.fit(X)
score_iso = -iso.score_samples(X)  # mayor = más anómalo
print("score range:", score_iso.min().round(3), score_iso.max().round(3))


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Autoencoder simple sin dependencias pesadas.


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
ae = MLPRegressor(hidden_layer_sizes=(8, 4, 8), max_iter=400, random_state=SEED).fit(Xs, Xs)
recon = ae.predict(Xs)
score_ae = np.mean((Xs - recon) ** 2, axis=1)
print("AE score range:", score_ae.min().round(3), score_ae.max().round(3))


## 13. Visualizaciones explicativas

ROC de cada modelo.


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
fpr_i, tpr_i, _ = roc_curve(y, score_iso)
fpr_a, tpr_a, _ = roc_curve(y, score_ae)
auc_i = roc_auc_score(y, score_iso)
auc_a = roc_auc_score(y, score_ae)
plt.figure(figsize=(6, 5))
plt.plot(fpr_i, tpr_i, label=f"IF AUC={auc_i:.3f}", color="#3F51B5")
plt.plot(fpr_a, tpr_a, label=f"AE AUC={auc_a:.3f}", color="#FF5722")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.title("ROC HVAC anomaly")
plt.tight_layout()
print({"AUC_IF": auc_i, "AUC_AE": auc_a})


## 14. Validaciones

Ambos AUC > 0.7 sobre el mock.


In [ ]:
assert auc_i > 0.7 and auc_a > 0.7


## 15. Errores comunes

1. **Entrenar sobre todo y evaluar sobre todo**: contaminación.
2. **Threshold fijo**: usar percentil sobre el train.
3. **Métricas de clasificación con desbalance** sin balancear.


## 16. Ejercicios propuestos

1. Implementa LOF (`LocalOutlierFactor`) y compara.
2. Añade SHAP a IF para explicar 5 fallos.
3. Ajusta `contamination` y observa el efecto.


## 17. Cómo se reutiliza con datos reales

Re-entrenar mensualmente con los datos del último mes (drift). El pipeline mismo no cambia.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `03_case_C_hvac_anomaly_detection/05_validacion_fallos_etiquetados.ipynb`.
- Documento web del caso: `docs/validation/ml-validation.md`.
